# 知能情報基礎演習

以下のリンクにアクセスすると、このノートブックを直接Google Colaboratoryで開くことができます。<br/>
適切な場所にコピーして使ってください。

https://colab.research.google.com/github/kawanishi-lab/kiso-enshu/blob/main/human-foundation-model/Sapiens_enshu.ipynb

# 人物画像の基盤モデルによる姿勢・深度・領域の推定

**人物画像の基盤モデル Sapiens** を実際に動かし、
1枚の人物画像から **姿勢・パーツ分割・深度・法線** の4つを推定して、
どんな条件でうまくいくかを調べる。

**Google Colab で動かす。GPU が必要である。** 使い方は 0-1 で説明する。

## このノートブックの構成

| 節 | 内容 | 区分 |
|----|------|------|
| 0 | 準備（GPU確認・モデルの取得・共通関数・動かし方） | 準備 |
| 1 | 4つの出力を並べて可視化する | **【標準】** |
| 2 | 出力の中身を数値として確かめる | **【標準】** |
| 3 | 人が写っていない画像を入れてみる | **【標準】** |
| 4 | 条件を変えて撮り、3段階で判定する | **【標準】** ← **ここまでで【標準】は完了** |
| 5 | うまくいかない理由を説明する | **【発展】** |
| 6 | さらに調べたい人へ（応用の実装を含む） | **【発展】**（1つ以上選ぶ） |

**【標準】は 1〜4 節、【発展】は 5〜6 節である。**
区切りのところに、進み具合を確かめるためのセルを置いてある。

## この演習でやること

**セルを上から順に実行すれば進められる。** 標準課題のコードは最初から全部書いてあるので、
プログラミングでつまずいて先に進めない、ということにはならない。

自分たちで決めるのは、**どんなデータを入れるか**である。

- どんな条件で写真を撮るか（4節）
- 担当する条件をどう分けるか（4節）
- 【発展】に進むなら、どう分析するか、どの選択課題をやるか（6節）

同じプログラムでも、**入れるデータを変えれば結果は変わる。**
その変わり方を観察して考察することが、この演習の中身である。

### ただし、コードはしっかり読むこと

実行するだけで数字と図は出る。しかし**中身を知らずに出た数字は、考察に使えない。**

**観察と考察に時間を使うこと。**

## 撮影のときに必ず守ること

- **被写体は原則として班のメンバー自身とする。**
- 第三者を撮る場合は、口頭で説明して同意を得る。説明する内容は次の3つ。
  「授業の課題で使う」「レポートと発表で使う」「**Google のクラウド（Colab）にアップロードする**」
- **背景に通行人が写り込んだら、撮り直すこと。**
- 画像の置き場所を1か所に決める（班の共有ドライブ）。個人のスマホやLINEに残さない。
- 発表やレポートに人物画像を載せる場合は、基本的には**顔が分からないように加工すること**

## ライセンス

- Sapiens の公開モデルは **CC BY-NC 4.0**。次の2点に注意する。
  - **BY（表示）**：出典を書く義務がある。レポートに
    「Sapiens (Khirodkar et al., ECCV 2024), facebook/sapiens-*-0.3b-torchscript, CC BY-NC 4.0」と明記する。
  - **NC（非営利）**：授業とレポートは問題ない。**企業インターンの成果物・受託業務・広告・有償公開には使えない。**
    深度や法線などの**出力画像も同じ扱い**になる。
- 論文: Khirodkar et al., *Sapiens: Foundation for Human Vision Models*, ECCV 2024.
- 実装: https://github.com/facebookresearch/sapiens

---
# 0. 準備

## 0-1. GPU を有効にする

このノートブックは **GPU がないと実用的な速さでは動かない**（1枚に数分かかる）。
最初に必ず GPU を有効にすること。**設定していないと、次のセルはエラーで止まる。**

1. 画面上部のメニューの **「ランタイム」** をクリック
2. **「ランタイムのタイプを変更」** を選ぶ
3. **「ハードウェア アクセラレータ」** で **「T4 GPU」**（または GPU）を選ぶ
4. **「保存」** を押す。ランタイムが再起動する

> **無料枠では GPU の割り当てに制限がある。** 締切直前は混むので、早めに取りかかること。
> 「GPU に接続できません」と出たら、しばらく待つか、時間をおいて試すこと。

設定できたか、下のセルで確かめる。

In [ ]:
import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU が有効になっていません。\n"
        "ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU\n"
        "を選び直してから、このセルをもう一度実行すること。\n"
        "（CPU のままだと 1 枚に数分かかり、演習が終わりません）")

## 0-2. 必要なライブラリと日本語フォントを入れる

**グラフに日本語を出すための設定も、ここでしておく**（やらないと図の文字が □ になる）。

In [ ]:
!pip -q install huggingface_hub japanize-matplotlib

import cv2, numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import torch.nn.functional as F

font_ok = False
try:
    import japanize_matplotlib
    font_ok = True
except Exception:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run("apt-get -qq install -y fonts-ipafont-gothic", shell=True)
    try:
        fm._load_fontmanager(try_read_cache=False)
    except Exception:
        pass
    for cand in ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP"]:
        if any(f.name == cand for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = cand; font_ok = True; break
plt.rcParams["axes.unicode_minus"] = False
print("日本語フォント:", "OK" if font_ok else "設定できず（図の日本語が □ になる）")

## 0-3. モデルを取得する

4つのタスクそれぞれに専用のモデルがある。ここでは最も小さい **0.3B** 版（TorchScript 形式）を使う。
合計で 5GB 程度をダウンロードするので数分かかる。

**この演習ではモデルを変えないこと。** 途中で変えると条件が揃わず、結果が比べられなくなる。

In [ ]:
from huggingface_hub import hf_hub_download

CHECKPOINTS = {
    "pose":   ("facebook/sapiens-pose-0.3b-torchscript",   "sapiens_0.3b_goliath_best_goliath_AP_573_torchscript.pt2"),
    "seg":    ("facebook/sapiens-seg-0.3b-torchscript",    "sapiens_0.3b_goliath_best_goliath_mIoU_7673_epoch_194_torchscript.pt2"),
    "depth":  ("facebook/sapiens-depth-0.3b-torchscript",  "sapiens_0.3b_render_people_epoch_100_torchscript.pt2"),
    "normal": ("facebook/sapiens-normal-0.3b-torchscript", "sapiens_0.3b_normal_render_people_epoch_66_torchscript.pt2"),
}
TASKS = ["pose", "seg", "depth", "normal"]

ckpt_path = {}
for task in TASKS:
    repo, fname = CHECKPOINTS[task]
    print("downloading:", task, "...")
    ckpt_path[task] = hf_hub_download(repo_id=repo, filename=fname)
print("\n完了")

## 0-4. 共通関数

**中身を読んでおくこと。** 何をしているか分からないまま結果だけ眺めても、考察は書けない。

前処理は公式の実装に合わせてある。

- 入力サイズは **高さ1024 × 幅768** に固定。元画像は縦横比を無視してこのサイズに引き伸ばされる。
- 画素値は平均 `[123.5, 116.5, 103.5]`、標準偏差 `[58.5, 57.0, 57.5]` で正規化する。

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IN_H, IN_W = 1024, 768
MEAN = np.array([123.5, 116.5, 103.5], dtype=np.float32)
STD  = np.array([ 58.5,  57.0,  57.5], dtype=np.float32)
MAX_SIDE = 1280      # 読み込み時に長辺をこの画素数まで縮める

_models = {}
def get_model(task):
    # 必要になったときだけモデルを GPU に載せる
    if task not in _models:
        print(f"loading {task} ...")
        _models[task] = torch.jit.load(ckpt_path[task]).to(DEVICE).eval()
    return _models[task]

def load_image(path, max_side=MAX_SIDE):
    # 画像を読み込んで RGB にする。大きすぎる写真は縮める
    # （縮めないと Colab のメモリが足りなくなる。モデルはどのみち 768x1024 に潰す）
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(
            f"{path} を読み込めませんでした。\n"
            "  ・iPhone の HEIC 形式ではありませんか（設定→カメラ→フォーマット→互換性優先）\n"
            "  ・ファイル名に使えない文字が入っていませんか")
    h, w = bgr.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1.0:
        bgr = cv2.resize(bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def preprocess(rgb):
    x = cv2.resize(rgb, (IN_W, IN_H), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    x = (x - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1))[None].to(DEVICE)

@torch.no_grad()
def raw_forward(task, rgb, to_cpu=True):
    # モデルの生の出力を返す（形を確認したいときに使う）
    out = get_model(task)(preprocess(rgb))
    if isinstance(out, (list, tuple)):
        out = out[0]
    out = out.float()
    return out.cpu() if to_cpu else out

def _resize_to_image(t, rgb):
    H, W = rgb.shape[:2]
    return F.interpolate(t, size=(H, W), mode="bilinear", align_corners=False).cpu()

def predict_seg(rgb):
    # パーツ分割。画素ごとのクラス番号（0=背景）を返す
    logits = _resize_to_image(raw_forward("seg", rgb, to_cpu=False), rgb)
    return logits[0].argmax(0).numpy().astype(np.uint8)

def predict_depth(rgb):
    # 深度。画素ごとの相対的な奥行きの値を返す（単位は無い）
    out = _resize_to_image(raw_forward("depth", rgb, to_cpu=False), rgb)
    return out[0, 0].numpy()

def predict_normal(rgb):
    # 法線。画素ごとの3次元ベクトル (H, W, 3) を返す
    out = _resize_to_image(raw_forward("normal", rgb, to_cpu=False), rgb)
    n = out[0].permute(1, 2, 0).numpy()
    return n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)

def predict_pose(rgb):
    # 姿勢。各キーポイントの画像座標 (K,2) とスコア (K,) を返す。
    # スコアはヒートマップのピーク値そのもので、0〜1 に較正されてはいない。
    # 公式実装は人物検出器で切り出してから推定するが、ここでは簡略化して
    # 画像全体をそのまま入力している。人物が画面いっぱいに写っているほど当たりやすい。
    hm = raw_forward("pose", rgb)[0].numpy()   # (K, h, w) のヒートマップ
    K, h, w = hm.shape
    H, W = rgb.shape[:2]
    flat = hm.reshape(K, -1)
    ys, xs = np.divmod(flat.argmax(axis=1), w)
    kps = np.stack([xs * (W / (w - 1)), ys * (H / (h - 1))], axis=1).astype(np.float32)
    return kps, flat.max(axis=1)

print("device:", DEVICE)

### コードを読んで考えること（1）

上のセルの `load_image` と `preprocess` を読んで、以下の点を考えよ。

1. `load_image` は、大きすぎる写真をどうしているか。**なぜそうする必要があるのか。**
2. `preprocess` は、入力画像を最終的にどんな大きさにしているか。
   そのとき、元の写真の**縦横比はどうなるか**。
3. 2 の結果、3:4 ではない写真（横位置で撮った写真など）は、モデルにどう見えることになるか。

### パーツのクラス名

分割モデルは28個のクラスを出力する。並び順は公式実装の `GOLIATH_CLASSES` に従っている。
**0番が背景**であることだけは確実なので、「人物領域かどうか」は `seg > 0` で判定できる。

In [ ]:
GOLIATH_CLASSES = [
    "Background", "Apparel", "Face_Neck", "Hair", "Left_Foot", "Left_Hand",
    "Left_Lower_Arm", "Left_Lower_Leg", "Left_Shoe", "Left_Sock", "Left_Upper_Arm",
    "Left_Upper_Leg", "Lower_Clothing", "Right_Foot", "Right_Hand", "Right_Lower_Arm",
    "Right_Lower_Leg", "Right_Shoe", "Right_Sock", "Right_Upper_Arm", "Right_Upper_Leg",
    "Torso", "Upper_Clothing", "Lower_Lip", "Upper_Lip", "Lower_Teeth", "Upper_Teeth", "Tongue",
]
print(len(GOLIATH_CLASSES), "classes")

### 主要な関節の番号

姿勢のキーポイントは308点あり、その大半（70番以降）は顔である。
番号と部位の対応は公式実装で決まっているので、主要な関節だけを表にして配る。

**このモデルに頭頂の点はない**（いちばん上は眉間の70番）。したがって**身長は測れない**。
また、この308点は COCO（よく使われる17点の定義）と**先頭9点しか一致していない**。
**手首は 41番（右）と 62番（左）** である。

In [ ]:
GOLIATH_KP = {
    "鼻": 0, "左目": 1, "右目": 2,
    "左肩": 5,  "右肩": 6,
    "左肘": 7,  "右肘": 8,
    "左腰": 9,  "右腰": 10,
    "左膝": 11, "右膝": 12,
    "左足首": 13, "右足首": 14,
    "右手首": 41, "左手首": 62,     # ← COCO と違うので注意
    "首": 69,
}
print(len(GOLIATH_KP), "個の主要な関節")

### 人物領域を取り出す

分割の出力 `seg` は、画素ごとのクラス番号が入った表である。**0番が背景**なので、
0より大きい画素を選べば人物領域になる。
x
numpy の配列は `配列 > 0` と書くだけで、**すべての画素をまとめて比べられる**（`for` は要らない）。
結果は True と False が並んだ表になる。以降のほぼすべての処理でこの関数を使う。

In [ ]:
# seg（画素ごとのクラス番号）から、人物領域を True にした表を返す。
# numpy の配列は「配列 > 0」と書くだけで、全画素をまとめて比べられる（for は要らない）。

def person_mask(seg):
    return seg > 0


# --- 確認：[[False, True], [True, False]] と表示される ---
test = np.array([[0, 1],
                 [2, 0]])
print(person_mask(test).tolist())

### 可視化の関数

**このセルは実行するだけでよい。**

In [ ]:
rng = np.random.default_rng(0)
PALETTE = rng.integers(0, 255, size=(64, 3), dtype=np.uint8)
PALETTE[0] = 0   # 背景は黒
try:
    CMAP_DEPTH = matplotlib.colormaps["inferno"]
except Exception:
    CMAP_DEPTH = plt.get_cmap("inferno")

def colorize_seg(seg):
    return PALETTE[seg % len(PALETTE)]

def colorize_depth(depth, mask=None):
    # 人物領域の値の範囲で正規化して色を付ける
    d = depth.copy().astype(np.float32)
    has_person = (mask is not None) and (mask.sum() >= 10)
    m = mask if has_person else np.ones_like(d, dtype=bool)
    lo, hi = np.percentile(d[m], 1), np.percentile(d[m], 99)
    d = np.clip((d - lo) / (hi - lo + 1e-8), 0, 1)
    img = (CMAP_DEPTH(d)[..., :3] * 255).astype(np.uint8)
    if has_person:
        img[~mask] = 40
    return img

def colorize_normal(normal):
    return ((normal + 1.0) / 2.0 * 255).clip(0, 255).astype(np.uint8)

def draw_pose(rgb, kps, scores, thr=0.3, radius=3):
    img = rgb.copy()
    for (x, y), s in zip(kps, scores):
        if s >= thr:
            cv2.circle(img, (int(x), int(y)), radius, (255, 60, 60), -1)
    return img

def stretch_ratio(out):
    # 前処理での引き伸ばし量。モデルは必ず 幅768 x 高さ1024（縦横比 0.75）に潰す。
    # 1.0 なら歪みなし。1 より大きいと横に伸ばされ、小さいと縦に伸ばされる。
    H, W = out["seg"].shape
    return (W / H) / (IN_W / IN_H)

def pose_outside_ratio(out):
    # 主要な関節のうち、「分割で人物とされた領域」の外に出てしまった点の割合[%]。
    # 姿勢と分割は別々のモデルの出力なので、食い違えばどちらかが間違っている。
    kps, _ = out["pose"]
    mask = person_mask(out["seg"])
    H, W = mask.shape
    outside = 0
    for i in GOLIATH_KP.values():
        x, y = kps[i]
        xi, yi = int(round(x)), int(round(y))
        if not (0 <= yi < H and 0 <= xi < W and mask[yi, xi]):
            outside += 1
    return 100.0 * outside / len(GOLIATH_KP)

def predict_all(rgb):
    out = {}
    out["seg"]    = predict_seg(rgb)
    out["depth"]  = predict_depth(rgb)
    out["normal"] = predict_normal(rgb)
    out["pose"]   = predict_pose(rgb)
    return out

def show_all(rgb, results, title=""):
    # 元画像＋4つの出力を横に並べる
    kps, sc = results["pose"]
    mask = person_mask(results["seg"])
    panels = [
        ("入力画像", rgb),
        (f"姿勢（点 {int((sc>0.3).sum())} / {len(sc)}）", draw_pose(rgb, kps, sc)),
        ("パーツ分割", colorize_seg(results["seg"])),
        ("深度", colorize_depth(results["depth"], mask)),
        ("法線", colorize_normal(results["normal"])),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(19, 5))
    for ax, (name, img) in zip(axes, panels):
        ax.imshow(img); ax.set_title(name, fontsize=11); ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

print("OK")

## 0-5. 画像を置く

画像は **条件ごとにフォルダを分けて** 置く。

```
images/
  01_正面/      IMG_0001.jpg ...
  02_後ろ姿/    ...
  03_逆光/      ...
```

In [ ]:
import os, glob, shutil
from pathlib import Path

ROOT = Path("/content/images")     # 画像を置くフォルダ．画面の一番左の縦にアイコンが何個か並んでいるところから，フォルダのアイコンを選択する．
ROOT.mkdir(exist_ok=True, parents=True)

def list_images(d):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    found = sorted(sum([glob.glob(str(Path(d) / e)) for e in exts], []))
    heic = sorted(sum([glob.glob(str(Path(d) / e)) for e in ("*.heic", "*.HEIC")], []))
    if heic:
        print(f"!! {Path(d).name}: HEIC 形式が {len(heic)} 枚あります。読み込めません。")
        print("!! iPhone の 設定 → カメラ → フォーマット → 互換性優先 にしてから撮り直すこと。")
    return found

def list_conditions(root=None):
    root = Path(root or ROOT)
    return sorted([p for p in root.iterdir() if p.is_dir()])

print("画像を置くフォルダ:", ROOT)

画像の入れ方は2通りある。**方法B（ドライブ）を勧める。**
方法Aは手軽だが、Colab のセッションが切れると画像が消えるので撮り直しになる。

In [ ]:
# ---- 方法A: パソコンからアップロードする（1回だけ試すとき用）--------
# 下の CONDITION を、いま撮った条件の名前に書き換えてから実行する。
CONDITION = "01_正面"
(ROOT / CONDITION).mkdir(exist_ok=True, parents=True)

from google.colab import files
uploaded = files.upload()          # ファイル選択ダイアログが開く
for name in uploaded:
    shutil.move(name, ROOT / CONDITION / name)
print("\n保存先:", ROOT / CONDITION)

In [ ]:
# ---- 方法B（推奨）: Google ドライブを使う ----
# Colab はしばらく放置すると切れて、アップロードした画像もモデルも消える。
# ドライブに置いておけば消えない。班で1つ共有フォルダを作るとよい。
# 使うときは、下の3行のコメント（先頭の #）を外してから実行する。

# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = Path('/content/drive/MyDrive/enshu1_images'); ROOT.mkdir(parents=True, exist_ok=True)

ALL_PATHS = sum([list_images(c) for c in list_conditions()], []) + list_images(ROOT)
print(f"合計 {len(ALL_PATHS)} 枚")
for c in list_conditions():
    print(f"  {c.name:20s} {len(list_images(c)):3d} 枚")

---
## 0-6. 動かし方は2通りある

このノートブックでは、セルの中に Python を書いて動かしている。
しかし研究の現場では、**コマンドライン（CLI）から実行するスクリプト**の形にすることが多い。
実際、Sapiens の公式実装も CLI スクリプトとして配布されている。

```
python vis_seg.py   <モデルのファイル> --input <画像フォルダ> --output_root <出力先>
python vis_depth.py <モデルのファイル> --input <画像フォルダ> --output_root <出力先>
```

**このノートブックは、その中身を読みやすく Python で書き直したもの**である。
やっていることは同じで、書き方と結果の受け取り方が違う。

|  | CLI（スクリプト） | Python（このノートブック） |
|---|---|---|
| 書き方 | `python 何とか.py --input ...` | `predict_seg(rgb)` |
| 結果の受け取り方 | 画像ファイルとして保存される | **変数として受け取れる** |
| 向いている場面 | 何百枚もまとめて処理するとき | 数を数える・グラフにするとき |
| この演習では | ここで試すだけ | **1節から先はこちらを使う** |

数を数えたりグラフを描いたりするには結果を変数で受け取る必要があるので、
**この演習の本体は Python で書く。**

---
---

# ここから【標準】

**1節から4節までが【標準】である。** 全員がここまでを終わらせること。

---

# 1.【標準】4つの出力を並べて可視化する

> **課題** Sapiens を試し、4つの出力を確認せよ。
> 姿勢・パーツ分割・深度・法線が何を出しているのかを調査せよ。

初回はモデルの読み込みに時間がかかる（2回目以降は速い）。

In [ ]:
assert ALL_PATHS, "画像がありません。0-5 のセルで画像をアップロードすること。"

img_path = ALL_PATHS[0]        # 試しに使う1枚。何枚目にするかはここで変える
print("使う画像:", img_path)

In [ ]:
# img_path は 0-6 で決めたもの。別の画像にしたければここで変える
# img_path = ALL_PATHS[1]

rgb = load_image(img_path)
print(img_path, rgb.shape)

res = predict_all(rgb)
show_all(rgb, res, title=Path(img_path).name)

### コードを読んで考えること（2）

0-4 の `predict_pose` を読んで、次の点について考える。

1. `predict_pose` は、モデルの出力（ヒートマップ）から**どうやって点の座標を決めているか**。
2. コメントに「公式実装は人物検出器で切り出してから推定する」とある。
   このノートブックは**その段階を省いている**。省いたことで、どんな画像で不利になるか。
3. 2 をふまえると、「Sapiens は◯◯な写真に弱い」と書くとき、**何を確かめてから書くべきか**。

### 考えること

- 姿勢の点は、体のどこに置かれているか。ずれているとしたら、どの部位がずれているか。
- パーツ分割の境界は、実際の体の境界と一致しているか。服と体の境界はどう扱われているか。
- 深度の色は、手前と奥のどちらが明るいか。**これは自分で確かめること。**
  色の付け方はこちらで決めたものなので、値の大小と奥行きの向きの対応は保証されていない。
  （なお公式のデモは色を付ける前に値を反転させている。ネットの作例と明暗が逆でも間違いではない）
- 法線の色は何を表しているか。RGB の3成分が、それぞれどの方向に対応しているか。

---
# 2.【標準】出力の中身を数値として確かめる

画像として眺めるだけでなく、**配列の形と値の範囲**を確認しておく。
「何が出力されているか」を説明するには、この情報が要る。

In [ ]:
for task in TASKS:
    out = raw_forward(task, rgb)
    print(f"{task:7s} 生の出力の形: {tuple(out.shape)}   "
          f"値の範囲: [{out.min():.3f}, {out.max():.3f}]")

print()
seg = res["seg"]
print("分割で実際に現れたクラス:")
for i, cnt in zip(*np.unique(seg, return_counts=True)):
    name = GOLIATH_CLASSES[i] if i < len(GOLIATH_CLASSES) else f"class_{i}"
    print(f"  {i:3d}  {name:16s} {cnt:8d} 画素 ({100*cnt/seg.size:5.2f} %)")

print()
d = res["depth"]; mask = person_mask(seg)
print(f"深度  全体: [{d.min():.3f}, {d.max():.3f}]")
if mask.sum() > 0:
    print(f"      人物領域のみ: [{d[mask].min():.3f}, {d[mask].max():.3f}]  中央値 {np.median(d[mask]):.3f}")

kps, sc = res["pose"]
print(f"姿勢  キーポイント数: {len(kps)}   スコアの範囲 [{sc.min():.3f}, {sc.max():.3f}]   平均 {sc.mean():.3f}")
print("      ※ スコアは 0〜1 に較正されていない。確率としては読めない。")

---
# 3.【標準】人が写っていない画像を入れてみる

> **課題** 人が写っていない画像（風景・食べ物・動物）を入れ、何が起きるか記録せよ。

このモデルは人物画像だけで学習されている。学習した経験の外側に何を出すのかを見る。
`images/99_人以外/` というフォルダを作って画像を入れてから実行すること。

In [ ]:
NONHUMAN_DIR = ROOT / "99_人以外"
paths_nh = list_images(NONHUMAN_DIR)

if not paths_nh:
    print(f"{NONHUMAN_DIR} に画像がありません。")
    print("0-5 のセルで CONDITION='99_人以外' にしてアップロードすること。")
else:
    for p in paths_nh[:3]:
        r = load_image(p)
        show_all(r, predict_all(r), title=Path(p).name)

### 記録すること

- 人がいないのに、姿勢の点はどこに置かれたか。**スコアは下がったか、下がらなかったか。**
- 背景以外のクラスが出た場合、どの部位として誤認されたか。
- 深度や法線は、それらしい値を出したか。人がいなくても出力が「それらしい」ことは何を意味するか。

---
# 4.【標準】条件を変えて撮り、3段階で判定する

> **課題** 各自、様々な条件で撮影し、評価をせよ。各自、異なる条件を担当せよ。
> 条件ごとに「うまくいった／一部崩れた／破綻した」の3段階で判定せよ。
> 4つのタスクは別々に判定すること。同時に崩れるとは限らない。

## 4-1. 担当する条件を決める

| 条件のグループ | 例 |
|---|---|
| 向き・姿勢 | 正面／横向き／後ろ姿／しゃがむ／腕を組む |
| 写る範囲 | 全身／上半身／顔だけ／遠くに小さく |
| 明るさ | 順光／逆光／暗い室内／強い影 |
| 遮蔽 | 机や椅子で隠れる／人が重なる／カバンや傘を持つ |
| 服装 | 厚着／薄着／だぼだぼの服／帽子／マスク |
| 人以外の人型 | イラスト／人形・マネキン／ポスターの人物 |

In [ ]:
# ■ ここは自分たちで決める ■
# 自分が担当する条件のフォルダ名を、リストに並べる。
# 上の表から選ぶこと。班の他の人と重ならないようにすること。
CONDITIONS = ["01_正面", "02_後ろ姿", "03_逆光"]

assert len(CONDITIONS) >= 2 and all(isinstance(c, str) for c in CONDITIONS), \
    "条件のフォルダ名を、文字列で2つ以上書くこと"
for c in CONDITIONS:
    (ROOT / c).mkdir(exist_ok=True, parents=True)
print("担当する条件:", CONDITIONS)

## 4-2. 担当した条件の画像をまとめて処理する

**同じ被写体・同じ場所で、条件だけを変えて撮ること。** そうしないと比較にならない。
下のセルを実行すると、条件ごとに結果が並んで表示される。**これを目で見て判定する。**

In [ ]:
MAX_PER_CONDITION = 4          # 1条件あたり何枚まで表示するか

all_results = {}
for cname in CONDITIONS:
    ps = list_images(ROOT / cname)[:MAX_PER_CONDITION]
    if not ps:
        print(f"（{cname} に画像がありません）"); continue
    print("=" * 90); print(f"条件: {cname}   {len(ps)} 枚"); print("=" * 90)
    for p in ps:
        r = load_image(p)
        out = predict_all(r)
        all_results[(cname, Path(p).name)] = out
        show_all(r, out, title=f"{cname} / {Path(p).name}")

print(f"\n処理した画像: {len(all_results)} 枚")

## 4-3. 判定表を作って、目で見た結果を書き込む

**自動では判定できない。** 「崩れているかどうか」は人が見て決めるしかない。
下のセルが空欄の表を作るので、CSV をダウンロードして表計算ソフトで埋め、
またアップロードして 5節に進む。

判定は次の3つのどれかを書く。

| 記号 | 意味 |
|---|---|
| `OK` | ほぼ正しい |
| `一部崩れ` | 一部がおかしいが、全体としては形になっている |
| `破綻` | まったく使えない |

> **基準を先に決めること。** 「一部崩れ」と「破綻」の境目は人によってずれる。
> 例えば「主要な関節のうち1つでも体の外にあれば一部崩れ、半分以上外れていれば破綻」のように
> グループで決めて、レポートに書くこと。

In [ ]:
REC_CSV = Path("/content/判定表.csv")

rows = []
for (cname, fname), out in all_results.items():
    kps, sc = out["pose"]
    mask = person_mask(out["seg"])
    rows.append({
        "条件": cname, "ファイル名": fname,
        # --- 機械が計算できる手がかり（5節で使う）---
        "人物の占める割合%": round(float(mask.mean() * 100), 2),
        "引き伸ばし率": round(stretch_ratio(out), 3),
        "姿勢が分割の外%": round(pose_outside_ratio(out), 1),
        "姿勢_スコア平均": round(float(sc.mean()), 4),
        # --- ここから下は人が目で見て入れる ---
        "判定_姿勢": "", "判定_分割": "", "判定_深度": "", "判定_法線": "",
        "気づいたこと": "",
    })

df = pd.DataFrame(rows).sort_values(["条件", "ファイル名"]).reset_index(drop=True)
if REC_CSV.exists():
    print(f"!! {REC_CSV} は既にあります。**上書きしません。**")
    print("!! 作り直したいときは、いったんファイルを手で消してから実行すること。")
else:
    df.to_csv(REC_CSV, index=False, encoding="utf-8-sig")
    print(f"→ {REC_CSV} に保存した。ダウンロードして、班全員で判定を埋めること。")
df

### コードを読んで考えること（3）

判定表には、機械が計算した3つの列が入っている。
0-4 の `stretch_ratio` と `pose_outside_ratio` を読んで、次に答えよ。

1. `stretch_ratio` の値が **1.0 のとき**、その写真はどんな写真か。
   **1.0 より大きいとき**はどうか。
2. `pose_outside_ratio` は、**どの2つの出力**を突き合わせているか。
3. `pose_outside_ratio` が大きいとき、**なぜ「何かおかしい」と言えるのか**。
   正解（人が目で見た判定）を使っていないのに、である。

---
---

# ここまでが【標準】課題

**4節まで終われば、【標準】は完了である。**
次に進む前に、次のものがそろっているか確認すること。

| 節 | そろっているはずのもの |
|----|----------------------|
| 1 | 4つの出力を並べた図 |
| 2 | 出力の形と値の範囲を確かめた記録 |
| 3 | 人が写っていない画像を入れた結果 |
| 4 | 担当した条件ごとの結果の図と、**空欄のない判定表** |

**【標準】だけでも発表とレポートは成立する。** 時間が足りなければ、ここで止めてよい。
そのぶん、うまくいった条件・いかない条件の観察と考察を厚くすること。

---
---

# ここから【発展】課題

**【標準】が終わってから取りかかること。**

- **5節**は、うまくいかない理由を数字で説明する。
- **6節**は選択課題。4つのうち **1つ以上** を選ぶ。

---

# 5.【発展】うまくいかない理由を説明する

> **課題** どんな条件が得意／不得意かを調査し、うまくいく条件・いかない条件に分類せよ。
> **得意不得意の理由について考察せよ。**
>
> **ヒント：うまくいかない理由をどう説明するか。**
> 人間の感覚で「よくある人物写真」に近いか、遠いか。
> 数字で測れる手がかり（人物が画面に占める割合／前処理での引き伸ばし量／
> 姿勢の点が分割で人物とされた領域の外に出ていないか）。

4節で「うまくいく条件」と「いかない条件」が分かった。**ではその差はどこから来るのか。**

ひとつの仮説は、**「Sapiens が学習に使った画像に近い条件ほど、うまくいく」** が挙げられる。
Sapiens は大量の人物写真で学習されている。
そこによく出てくる写り方から遠ざかるほど、崩れやすいはず。
以下の手がかりをもとに確認してみましょう。

| # | 手がかり | どう測るか |
|---|----------|-----------|
| 1 | 人間の感覚 | その条件は「よくある人物写真」に近いか、珍しいか。自分で判断する |
| 2 | 人物の写り方 | 画面に占める割合、前処理での引き伸ばし量。 |
| 3 | 出力どうしの食い違い | 姿勢の点が、分割の人物領域の外に出ていないか。|



---
# 6.【発展】さらに調べたい人へ（1つ以上選ぶ）

ここから先は **選択課題** である。班で相談して1つ以上を選び、レポートに載せること。


## 6-A. 深度の値と、メジャーで測った距離を比べる

同じ人が 1m, 2m, 3m の位置に立った画像を撮り、人物領域の深度の中央値と距離の関係を調べる。
`images/距離_1.0m/` のようにフォルダ名に距離を入れておくと、下のセルが読み取る。
どのような関係になっているか、グラフ化すると良い。

## 6-B. 姿勢から身体寸法を求め、実測値と比べる

使う関節の番号は 0-4 の `GOLIATH_KP` にある。**頭頂の点はないので身長は測れない。**
測れるのは点と点の間の距離だけである。

画像から求まるのは **画素数** であって cm ではない。
長さの分かっている物（30cm定規、A4用紙の長辺29.7cm など）を体と同じ距離に置いて一緒に撮り、
その物の画素数から換算係数を出すこと。

## 6-C. 同じ条件で5回撮り、推定値のばらつきを求める

`images/ばらつき/` に、同じ人が同じポーズで写った画像を5枚以上入れてから実行する。

- 推定された距離はどの程度ばらつくか
- 検出した人の大きさはどの程度ばらつくか

## 6-D. パーツ分割で人物領域だけをぼかす（匿名化）

`seg > 0` が人物領域である。この領域だけを加工すればよい。

### 匿名化でなくてもよい

その他、使えそうな情報を以下に挙げる。

| 使う出力 | できること | 例 |
|---------|-----------|-----|
| パーツ分割 | 画素ごとに部位が分かる | 背景だけ差し替える／服の色を変える／体の部位ごとに色分けする |
| 深度 | 手前と奥の順序が分かる | 背景だけぼかす（ポートレート風）／人の前後関係で重ね順を決める |
| 姿勢 | 関節の位置が分かる | 関節角度を測る／姿勢が崩れたら知らせる／点の動きで動作を判定する |
| 法線 | 面の向きが分かる | 向きに応じて陰影を付け直す／体の凹凸を強調する |
| 組み合わせ | | 分割で人だけ切り出し、深度で前後に並べ替えて合成する |


> 人物画像を扱うので、**発表やレポートに載せる画像は顔が分からないように加工すること。**